# 五会话同状态标签：按日价格路径向量化身份与预算

固定2019-12-19最早40个合法分钟和+0.10 SOXX动作。将已修正的Qlib事件/HOLD五会话路径压成时间有序的NumPy价格平面，先比较每个事件的费用、股数、现金、终端NAV和终端可用mark回撤，过精确身份门再估计全量耗时。此卡不选价值高的分钟，也不训练模型。

In [1]:
import os,json,hashlib,warnings
from pathlib import Path
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
warnings.filterwarnings("ignore",message="The 'generic' unit for NumPy timedelta")
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from src.core.recorder_sources import load_verified_recorder_objects
from src.backtest.stage1_event_checkpoint import capture_after_open
from src.backtest.stage1_capfree_lock_episode import replay_capfree_lock_episode
from src.analysis.stage1_capfree_lock_vectorized import compile_lock_price_plane,value_lock_event
from src.backtest.ibkr_tiered import settle_tiered_event
from src.workflow.stage1_dated_controller import DatedModel,DatedStage1Controllers
from src.workflow.stage1_f696_training_reference import prepare as prepare_f696_source
from src.workflow.intraday_recovery_direct_allocation import IntradayRecoveryDirectAllocationSpec
from src.workflow.stage1_open_action_value import Stage1OpenActionValueSpec
from copy import deepcopy
root=Path.cwd();card=root/"research/stage1_v5_lock_vectorized_parity_20261030";plan=json.loads((card/"plan.json").read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
parent=root/plan["parent_card"];old=root/plan["old_source_card"]
assert sha(parent/"summary.json")==plan["parent_summary_sha256"] and sha(parent/"verification_manifest.json")==plan["parent_manifest_sha256"]
assert sha(old/"plan.json")==plan["old_source_plan_sha256"]
assert sha(root/plan["checkpoint_code"])==plan["checkpoint_code_sha256"]
assert sha(root/plan["reference_module"])==plan["reference_module_sha256"]
assert sha(root/plan["runtime_card"]/"audit.json")==plan["runtime_audit_sha256"]
assert sha(root/plan["minute_provider_uri"]/"qlib_store_manifest.json")==plan["minute_provider_manifest_sha256"]
parent_plan=json.loads((parent/"plan.json").read_text());old_plan=json.loads((old/"plan.json").read_text())
prepared=prepare_f696_source(root=root,card=old,plan=old_plan)
qlib.init(provider_uri=str(root/plan["minute_provider_uri"]),region="us",kernels=1)
names=["analysis/native/dated_f696_0","analysis/controller_audit",*[f"models/open_{y}" for y in parent_plan["signal_years"]],*[f"models/recovery_{y}" for y in (2020,2021,2022)]]
objects,receipt=load_verified_recorder_objects(parent,expected_recorder_id=plan["parent_recorder_id"],artifacts=names,expected_manifest_sha256=plan["parent_manifest_sha256"])
book=objects["analysis/native/dated_f696_0"];frame=prepared["frame"]
assert len(frame)==1056 and book["audit"]["status"]=="passed"
open_models={y:DatedModel(objects[f"models/open_{y}"],f"{y-1}-12-31",f"{parent_plan['action_recorder_id']}:open_action_value_models/{y}") for y in parent_plan["signal_years"]}
recovery_models={}
for y in parent_plan["signal_years"]:
 if y in parent_plan["explicit_no_recovery_model_years"]:recovery_models[y]=DatedModel(None,f"{y-1}-12-31",f"explicit_no_model_{y}_insufficient_prior_teacher",explicit_no_model=True)
 else:recovery_models[y]=DatedModel(objects[f"models/recovery_{y}"],f"{y-1}-12-31",f"{plan['parent_recorder_id']}:models/recovery_{y}")
controller=DatedStage1Controllers(calendar=frame[["signal_date","trade_date"]],open_models=open_models,recovery_models=recovery_models)
pd.testing.assert_frame_equal(controller.audit(),objects["analysis/controller_audit"],check_dtype=False)
recovery_spec=IntradayRecoveryDirectAllocationSpec(root=root,**prepared["action_plan"]["recovery_spec"])
action_spec=Stage1OpenActionValueSpec(root=root,**prepared["action_plan"]["action_spec"])
print('dated parent and minute source verified',len(receipt))

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[27796:MainThread](2026-10-30 14:09:18,262) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[27796:MainThread](2026-10-30 14:09:18,264) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[27796:MainThread](2026-10-30 14:09:18,264) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/full_path_tiered_allocation_20261013')}


[27796:MainThread](2026-10-30 14:09:18,341) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[27796:MainThread](2026-10-30 14:09:18,342) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[27796:MainThread](2026-10-30 14:09:18,342) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_event_guard_fullminute_20261028')}


dated parent and minute source verified 13


In [2]:
import time
from src.backtest.event_guard_clock import guard_slot
day=pd.Timestamp(plan['event_day']);checkpoint=capture_after_open(frame,event_day=day,controller_by_date=controller,recovery_spec=recovery_spec,action_spec=action_spec,selection=prepared['selection'],plan=parent_plan)
last=frame.iloc[checkpoint.index+4]
rawbars=D.features(['SOXL','SOXX'],['$open','$close','$raw_open','$price_valid'],start_time=str(day.date())+' 09:31:00',end_time=str(pd.Timestamp(last.trade_date).date())+' 16:00:00',freq='1min')
parts=[]
for symbol in ('SOXL','SOXX'):
 x=rawbars.xs(symbol,level='instrument').copy();x.columns=[symbol+'_'+k.removeprefix('$') for k in x.columns];parts.append(x)
quotes=pd.concat(parts,axis=1).sort_index();assert quotes[['SOXL_price_valid','SOXX_price_valid']].eq(1).all().all()
first=quotes.loc[quotes.index.normalize()==day]
clocks=first.index[(first.index>=day+pd.Timedelta(hours=9,minutes=48)) & (first.index<day+pd.Timedelta(hours=10,minutes=29))][:plan['sample_events']]
assert len(clocks)==plan['sample_events']
compile_start=time.perf_counter();plane=compile_lock_price_plane(frame.iloc[checkpoint.index:checkpoint.index+5],quotes);compile_seconds=time.perf_counter()-compile_start
assert plane.prices.shape[1]==2 and plane.stamps[-1]==pd.Timestamp(last.exit_date).replace(hour=9,minute=47).to_datetime64()
print('compiled chronological Qlib price plane',len(plane.stamps),'marks')

compiled chronological Qlib price plane 1758 marks


In [3]:
events=[]
for stamp in clocks:
 mark=first.loc[stamp];price=np.array([mark.SOXL_close,mark.SOXX_close],float)
 nav=float(checkpoint.state.q@price+checkpoint.state.cash);w=np.r_[checkpoint.state.q*price,checkpoint.state.cash]/nav
 slot=guard_slot(stamp,session_open=day+pd.Timedelta(hours=9,minutes=30),session_close=first.index.max(),original_open_fill=day+pd.Timedelta(hours=9,minutes=47))
 fillbar=first.loc[slot.fill_at+pd.Timedelta(minutes=1)]
 events.append({'trade_date':day,'bar_end_at':stamp,'decision_at':slot.decision_at,'fill_at':slot.fill_at,'desired_weights':w+np.array([0.,.10,-.10]),'adjusted_fill_prices':np.array([fillbar.SOXL_open,fillbar.SOXX_open],float),'raw_fill_prices':np.array([fillbar.SOXL_raw_open,fillbar.SOXX_raw_open],float)})
vector_start=time.perf_counter()
fast=[value_lock_event(plane,bar_end_at=e['bar_end_at'],fill_at=e['fill_at'],quantities=checkpoint.state.q,cash=checkpoint.state.cash,starting_peak=checkpoint.state.peak,fees=checkpoint.state.fees,desired_weights=e['desired_weights']) for e in events]
vector_seconds=time.perf_counter()-vector_start
reference=[replay_capfree_lock_episode(frame,checkpoint,quotes,event=e,horizon_sessions=5) for e in events]
max_error={'terminal_nav':0.,'max_drawdown':0.,'cash':0.,'quantity':0.,'fee':0.}
for x,y in zip(fast,reference):
 assert x['label_available_at']==pd.Timestamp(last.exit_date)+pd.Timedelta(hours=9,minutes=49)
 for arm in ('HOLD','EVENT'):
  a,b=x[arm],y[arm]
  max_error['terminal_nav']=max(max_error['terminal_nav'],abs(a['terminal_nav']-float(b['path'].iloc[-1].next_open_nav)))
  max_error['max_drawdown']=max(max_error['max_drawdown'],abs(a['max_drawdown']-b['terminal_available_mark_max_drawdown']))
  max_error['cash']=max(max_error['cash'],abs(a['cash']-b['ending_cash']))
  max_error['quantity']=max(max_error['quantity'],float(np.max(np.abs(a['quantities']-b['ending_quantities']))))
  max_error['fee']=max(max_error['fee'],abs(a['fee_usd']-sum(o['total'] for o in b['orders'])))
  assert a['monthly_fee_shares']==b['monthly_fee_shares']
 assert len(x['EVENT']['orders'])==1
assert max_error['terminal_nav']<=plan['maximum_terminal_nav_error_usd'] and max_error['max_drawdown']<=plan['maximum_maxdrawdown_error']
assert max_error['cash']==max_error['quantity']==max_error['fee']==0
projected=compile_seconds*1056+vector_seconds/len(events)*plan['full_mature_path_rows']
audit={'experiment':plan['experiment'],'plan_sha256':sha(card/'plan.json'),'vector_module_sha256':sha(root/'src/analysis/stage1_capfree_lock_vectorized.py'),'reference_module_sha256':plan['reference_module_sha256'],'sample_events':len(events),'compiled_marks':len(plane.stamps),'compile_seconds':compile_seconds,'vectorized_seconds':vector_seconds,'vectorized_seconds_per_event':vector_seconds/len(events),'projected_full_mature_rows_seconds_single_arm':projected,'max_parity_errors':max_error,'single_arm_projection_within_1800_seconds':bool(projected<=plan['maximum_projected_full_seconds']),'new_model_fits':0,'new_candidate_accounts':0,'historical_research_passed':False,'decision':'vectorized_one_arm_parity_pass_budget_conditional' if projected<=plan['maximum_projected_full_seconds'] else 'vectorized_one_arm_parity_pass_runtime_budget_failed'}
(card/'audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2)+'\n')
print(json.dumps(audit,ensure_ascii=False,indent=2))

{
  "experiment": "stage1_v5_lock_vectorized_parity_20261030",
  "plan_sha256": "322938c0077551329ffd7d7fcf1e833308a390268320e40ebba5e16c55efe249",
  "vector_module_sha256": "348373d8e50328e4cb382a74a7a17a958321e68a3a1ea3c8fa9f830e820a3d4f",
  "reference_module_sha256": "8654a7c8adff7d5eff67013e24b0b299eb24c2d5cd5175c8676ba4787672819d",
  "sample_events": 40,
  "compiled_marks": 1758,
  "compile_seconds": 0.003284792008344084,
  "vectorized_seconds": 0.01272774999961257,
  "vectorized_seconds_per_event": 0.00031819374999031425,
  "projected_full_mature_rows_seconds_single_arm": 125.33567383210175,
  "max_parity_errors": {
    "terminal_nav": 0.0,
    "max_drawdown": 0.0,
    "cash": 0.0,
    "quantity": 0.0,
    "fee": 0.0
  },
  "single_arm_projection_within_1800_seconds": true,
  "new_model_fits": 0,
  "new_candidate_accounts": 0,
  "historical_research_passed": false,
  "decision": "vectorized_one_arm_parity_pass_budget_conditional"
}


## 中文结论

SOXX：零新拟合/账户。固定2019-12-19最早40个合法分钟，Qlib五会话时间有序价格平面1758个mark；向量化SOXX单臂40次耗时0.0127秒、约0.318毫秒/事件，连同每会话编译的单臂全量外推约125秒。与参考逐事件回放的HOLD/EVENT终端NAV、可用mark最大回撤、股数、现金、Tiered费和月度计数逐事件差0。此结果验证普通早盘一臂核心，不证明恢复后和半日市、两臂及382996全量真实耗时；下一步做四类机械时钟与SOXL arm的同域对账，才准入批量Recorder。2021/2022价值未用。